# Graded unseen-flux tests for the 1D cubic checkpoints (ICLR 2027 rebuttal, 1b)

This notebook evaluates the submitted 1D cubic checkpoints of HFluxNO, DPOT and DISCO on the test sets written by `scripts/data/generate_unseen_flux_1d.py`. The metrics and the segment construction are those of `evaluate_cubic.ipynb`, with three differences.

- **Window starts.** A trajectory of 99 frames has 60 windows of 40 frames. The submitted tables use all 60. This notebook uses the 10 starts 2, 8, ..., 56, after the first section has checked on the in-distribution set that both choices give the same mean.
- **Persistence.** Every set is also scored with the prediction that holds the last context frame.
- **Non-finite rollouts.** A set reports the fraction of segments with a non-finite metric and the mean over the remaining segments. The mean of `evaluate_cubic.ipynb` is NaN as soon as one segment is.

Each set produces one JSON file in `data/analysis/unseen_flux/`. Besides the means, the file holds the mean error per coefficient draw and per window start.

In [1]:
import json
import warnings
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from jaxtyping import Array, Bool, Float
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints/cubicflux_1d")
test_dir = datadir / "datasets/cubic_no_source/data/test"
output_dir = datadir / "analysis/unseen_flux"
output_dir.mkdir(parents=True, exist_ok=True)

jax.config.update("jax_default_device", jax.devices("gpu")[3])

In [2]:
# The checkpoints of the ICLR 2027 submission, as listed in evaluate_cubic.ipynb.
model_paths = {
    "DPOT": [
        "seed=0/26-04-10-14_49_05",
        "seed=10/26-04-16-18_37_37",
        "seed=20/26-04-17-07_57_49",
    ],
    "HyperFluxFNOLocal": [
        "seed=0/26-04-16-00_31_47",
        "seed=10/26-04-17-07_57_49",
        "seed=20/26-04-17-10_50_08",
    ],
    "DISCO": [
        "seed=0/26-04-10-21_26_22",
        "seed=10/26-04-16-18_37_50",
        "seed=20/26-04-16-18_37_50",
    ],
}

CONTEXT_LENGTH = 20
WINDOW_LENGTH = 40  # 20 context frames and a 20-step rollout
NUM_FRAMES = 99  # frames 0..98 of each trajectory, as in evaluate_cubic.ipynb
ALL_WINDOW_STARTS = np.arange(NUM_FRAMES - WINDOW_LENGTH + 1)  # 0, 1, ..., 59
WINDOW_STARTS = np.arange(2, 60, 6)  # 2, 8, ..., 56
# The checkpoints were trained on the first 80 frames. A window lies inside that
# range when its last frame does.
NUM_TRAINED_FRAMES = 80
BATCH_SIZE = 6000

# name -> (file name, stride that brings the stored frames to dt = 0.005).
# The sets of the submission store all 1001 solver frames; the new sets store every 10th.
SUBMITTED_SETS = {
    "cubic_in_distribution": ("cubic_no_source_large_test_seed=10.hdf5", 10),
    "sine_box_submitted_grf": ("sine_1d_test_grf_seed=12.hdf5", 10),
    "sine_box_submitted_step": ("sine_1d_test_step_seed=12.hdf5", 10),
}
NEW_SETS = {
    f"{name}_{ic}": (f"{name}_test_{ic}_seed={seed}.hdf5", 1)
    for name, seed in [
        ("sine_box_godunov", 12),
        ("sine_rescaled_1-2", 13),
        ("sine_rescaled_2-3", 13),
        ("sine_rescaled_3-4", 13),
        ("cubic_shell_1-1.5", 13),
        ("cubic_shell_1.5-2", 13),
        ("cubic_shell_2-2.5", 13),
    ]
    for ic in ("grf", "step")
}

## Loading and metrics

In [3]:
def load_test_set(
    name: str,
) -> tuple[
    Float[np.ndarray, "pde ic {NUM_FRAMES} 1 x"],
    Bool[np.ndarray, "pde ic"],
    xr.Dataset,
    tuple[float, float],
]:
    """Load a test set and return `(values, failed, dataset, (dt, dx))`.

    `failed` marks the trajectories whose reference solve failed. Sets written before
    `generate_unseen_flux_1d.py` have no such record and none is marked.
    """
    filename, time_stride = (SUBMITTED_SETS | NEW_SETS)[name]
    dataset = xr.open_dataset(test_dir / filename, engine="h5netcdf", chunks={})
    dataset = dataset.isel(t=slice(None, None, time_stride)).isel(
        t=slice(0, NUM_FRAMES)
    )
    if dataset.sizes["t"] != NUM_FRAMES:
        raise ValueError(f"{filename} has {dataset.sizes['t']} frames after striding.")
    dt = float(dataset["t"][1] - dataset["t"][0])
    dx = float(dataset["x"][1] - dataset["x"][0])
    values = dataset["values"].values.astype(np.float32)
    if "failed" in dataset:
        failed = dataset["failed"].values.astype(bool)
    else:
        failed = np.zeros(values.shape[:2], dtype=bool)
    return values, failed, dataset, (dt, dx)


def make_segments(
    values: Float[np.ndarray, "pde ic time 1 x"], window_starts: np.ndarray
) -> Float[np.ndarray, "pde ic window {WINDOW_LENGTH} 1 x"]:
    """Cut the windows of `WINDOW_LENGTH` frames that begin at `window_starts`."""
    frame_indices = window_starts[:, None] + np.arange(WINDOW_LENGTH)
    return values[:, :, frame_indices]

In [4]:
# Copied from evaluate_cubic.ipynb without changes.
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }


@jax.jit
def compute_persistence_metrics(
    u: Float[Array, "{WINDOW_LENGTH} 1 x"],
) -> dict[str, Float[Array, ""]]:
    """The metrics of `compute_metrics` for the prediction that holds the last context
    frame over the whole rollout."""
    context, u_data = u[:CONTEXT_LENGTH], u[CONTEXT_LENGTH:]
    u_pred = jnp.broadcast_to(context[-1], u_data.shape)
    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }


def evaluate_segments(
    metrics_fn, segments: Float[np.ndarray, "pde ic window time 1 x"]
) -> dict[str, Float[np.ndarray, "pde ic window"]]:
    """Apply `metrics_fn` to every segment and keep the (pde, ic, window) layout."""
    leading_shape = segments.shape[:3]
    flat = segments.reshape(-1, *segments.shape[3:])
    results = jax.lax.map(metrics_fn, flat, batch_size=BATCH_SIZE)
    return {k: np.asarray(v).reshape(leading_shape) for k, v in results.items()}


def summarize(
    metrics: dict[str, Float[np.ndarray, "pde ic window"]],
    failed: Bool[np.ndarray, "pde ic"],
    window_starts: np.ndarray,
) -> dict:
    """Reduce per-segment metrics to the numbers that are reported.

    Segments of a trajectory whose reference solve failed are left out of everything.
    Among the rest, a segment is non-finite when any of its metrics is. The summary
    holds the fraction of such segments and, for every metric, the mean over the finite
    segments: over all windows (`mean`), over the windows inside the trained time range
    (`mean_trained_range`), and per coefficient draw and window start
    (`per_draw_window`, with NaN where a cell has no finite segment).
    """
    has_reference = np.broadcast_to(
        ~failed[:, :, None], next(iter(metrics.values())).shape
    )
    is_finite = has_reference & np.all(
        [np.isfinite(m) for m in metrics.values()], axis=0
    )
    in_trained_range = window_starts + WINDOW_LENGTH <= NUM_TRAINED_FRAMES

    summary = {
        "num_segments": int(has_reference.sum()),
        "fraction_nonfinite": float(1.0 - is_finite.sum() / has_reference.sum()),
    }
    with np.errstate(invalid="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        for name, m in metrics.items():
            masked = np.where(is_finite, m, np.nan)
            summary[name] = {
                "mean": float(np.nanmean(masked)),
                "mean_trained_range": float(np.nanmean(masked[:, :, in_trained_range])),
                "per_draw_window": np.nanmean(masked, axis=1).tolist(),
            }
    return summary


def load_checkpoint(model_type: str, path: str):
    model = load_model(checkpoint_dir / model_type / "OneStepLoss" / path)
    return eqx.nn.inference_mode(model, True)


def evaluate_set(name: str, window_starts: np.ndarray) -> dict:
    """Evaluate every checkpoint and the persistence prediction on one test set."""
    values, failed, dataset, (dt, dx) = load_test_set(name)
    segments = make_segments(values, window_starts)

    results = {
        "Persistence": [
            summarize(
                evaluate_segments(compute_persistence_metrics, segments),
                failed,
                window_starts,
            )
        ]
    }
    for model_type, paths in model_paths.items():
        results[model_type] = []
        for path in tqdm(paths, desc=f"{name} / {model_type}"):
            model = load_checkpoint(model_type, path)
            metrics = evaluate_segments(
                eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))), segments
            )
            results[model_type].append(summarize(metrics, failed, window_starts))

    return {
        "set": name,
        "file": (SUBMITTED_SETS | NEW_SETS)[name][0],
        "window_starts": window_starts.tolist(),
        "param": [str(p) for p in dataset["param"].values],
        "coeffs": dataset["coeffs"].values.tolist(),
        "num_failed_reference": int(failed.sum()),
        "results": results,
    }


def mean_and_std(result: dict, model_type: str, metric: str, key: str = "mean"):
    """Mean and standard deviation over the seeds of one model."""
    per_seed = [summary[metric][key] for summary in result["results"][model_type]]
    return float(np.mean(per_seed)), float(np.std(per_seed))

## Check 1: the checkpoints load, and 10 window starts reproduce the 60-window mean

The in-distribution set is evaluated once at all 60 window starts. The 60-window means must reproduce Table 1 of the submission (relative l2 over the rollout: DPOT 1.36e-1, DISCO 8.22e-2, HFluxNO 5.21e-2). The 10-window means are read from the same per-segment results. The new sets use 10 windows if, for every model, the two means differ by less than the standard deviation over seeds.

In [5]:
values, failed, _, (dt, dx) = load_test_set("cubic_in_distribution")
segments = make_segments(values, ALL_WINDOW_STARTS)
is_subsampled = np.isin(ALL_WINDOW_STARTS, WINDOW_STARTS)

window_check = {}
for model_type, paths in model_paths.items():
    window_check[model_type] = []
    for path in tqdm(paths, desc=model_type):
        model = load_checkpoint(model_type, path)
        metrics = evaluate_segments(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))), segments
        )
        window_check[model_type].append(
            {
                "all": summarize(metrics, failed, ALL_WINDOW_STARTS),
                "subsampled": summarize(
                    {k: v[:, :, is_subsampled] for k, v in metrics.items()},
                    failed,
                    WINDOW_STARTS,
                ),
            }
        )
del segments

for metric in ("l2_rollout", "l_inf_rollout", "l2_onestep", "l_inf_onestep"):
    print(metric)
    for model_type, per_seed in window_check.items():
        mean_all = np.asarray([s["all"][metric]["mean"] for s in per_seed])
        mean_sub = np.asarray([s["subsampled"][metric]["mean"] for s in per_seed])
        difference = abs(mean_sub.mean() - mean_all.mean())
        print(
            f"  {model_type:18s} 60 windows: {mean_all.mean():.3e} ± {mean_all.std():.1e}"
            f"   10 windows: {mean_sub.mean():.3e} ± {mean_sub.std():.1e}"
            f"   difference / seed std: {difference / mean_all.std():.2f}"
        )

DPOT:   0%|          | 0/3 [00:00<?, ?it/s]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/wrapper.py:11: SyntaxWarning: invalid escape sequence '\D'
  u_{t_\Delta t} to a multiphysics model.
W1111 11:45:19.542206  491951 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
HyperFluxFNOLocal:   0%|          | 0/3 [00:00<?, ?it/s]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:73: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Outp

l2_rollout
  DPOT               60 windows: 1.360e-01 ± 3.1e-03   10 windows: 1.363e-01 ± 3.1e-03   difference / seed std: 0.08
  HyperFluxFNOLocal  60 windows: 5.207e-02 ± 2.1e-03   10 windows: 5.221e-02 ± 2.1e-03   difference / seed std: 0.07
  DISCO              60 windows: 8.213e-02 ± 5.6e-03   10 windows: 8.253e-02 ± 5.6e-03   difference / seed std: 0.07
l_inf_rollout
  DPOT               60 windows: 6.766e-01 ± 9.7e-03   10 windows: 6.773e-01 ± 9.7e-03   difference / seed std: 0.08
  HyperFluxFNOLocal  60 windows: 3.679e-01 ± 1.6e-02   10 windows: 3.685e-01 ± 1.6e-02   difference / seed std: 0.04
  DISCO              60 windows: 4.544e-01 ± 3.1e-02   10 windows: 4.558e-01 ± 3.1e-02   difference / seed std: 0.05
l2_onestep
  DPOT               60 windows: 7.398e-03 ± 9.1e-05   10 windows: 7.434e-03 ± 9.3e-05   difference / seed std: 0.39
  HyperFluxFNOLocal  60 windows: 4.101e-03 ± 1.1e-04   10 windows: 4.119e-03 ± 1.2e-04   difference / seed std: 0.16
  DISCO              60 wind

## Check 2: the control set solves the problems of the submitted sine set

`sine_box_godunov` with seed 12 repeats the key derivation of the submitted sine sets. If the draws are reproduced, frame 0 and the coefficients agree and the two reference solutions can be compared trajectory by trajectory. If they are not reproduced (a change in the waveform code or in the JAX version), the control set is still a valid sample of the same family, and only this comparison is lost.

In [6]:
for ic in ("grf", "step"):
    values_old, _, dataset_old, _ = load_test_set(f"sine_box_submitted_{ic}")
    values_new, failed_new, dataset_new, _ = load_test_set(f"sine_box_godunov_{ic}")
    same_coeffs = values_old.shape == values_new.shape and np.allclose(
        dataset_old["coeffs"].values, dataset_new["coeffs"].values, atol=1e-6
    )
    same_ics = same_coeffs and np.allclose(
        values_old[:, :, 0], values_new[:, :, 0], atol=1e-5, equal_nan=True
    )
    print(
        f"{ic}: same coefficients: {same_coeffs}, same initial conditions: {same_ics}"
    )
    if same_ics:
        ok = ~failed_new
        difference = np.linalg.norm(
            (values_old - values_new)[ok].reshape(ok.sum(), -1), axis=1
        ) / np.linalg.norm(values_new[ok].reshape(ok.sum(), -1), axis=1)
        mass_old = values_old.sum(axis=-1)
        drift_old = np.abs(mass_old - mass_old[:, :, :1]).max() / values_old.shape[-1]
        print(
            f"  relative l2 difference between the two references: "
            f"median {np.median(difference):.2e}, 90% {np.quantile(difference, 0.9):.2e}, "
            f"max {difference.max():.2e}"
        )
        print(f"  largest drift of the mean of u in the submitted set: {drift_old:.2e}")

grf: same coefficients: True, same initial conditions: True
  relative l2 difference between the two references: median 6.68e-05, 90% 2.48e-03, max 1.20e-01
  largest drift of the mean of u in the submitted set: 4.07e-02
step: same coefficients: True, same initial conditions: True
  relative l2 difference between the two references: median 2.89e-04, 90% 5.68e-03, max 3.69e-02
  largest drift of the mean of u in the submitted set: 6.02e-03


## Evaluation of all sets

Set `sets_to_run` to a subset for the pilot. A set whose JSON file exists is skipped, so delete the file to recompute it.

In [7]:
sets_to_run = ["sine_box_submitted_grf", "sine_box_submitted_step", *NEW_SETS]

for name in sets_to_run:
    output_path = output_dir / f"{name}.json"
    if output_path.exists():
        print(f"{output_path} exists, skipped")
        continue
    result = evaluate_set(name, WINDOW_STARTS)
    with open(output_path, "w") as f:
        json.dump(result, f)

sine_box_submitted_grf / HyperFluxFNOLocal:   0%|          | 0/3 [00:00<?, ?it/s]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:73: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector
sine_box_submitted_grf / HyperFluxFNOLocal:  33%|███▎      | 1/3 [01:44<03:29, 104.81s/it]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:73: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/s

## Error against the bin

The table lists the 20-step relative l2 and l-infinity errors as mean ± standard deviation over the three seeds, the persistence error, and the largest fraction of non-finite segments among the seeds.

In [9]:
all_results = {}
for name in sets_to_run:
    with open(output_dir / f"{name}.json") as f:
        all_results[name] = json.load(f)

for metric in ("l2_rollout", "l_inf_rollout"):
    print(metric)
    for name, result in all_results.items():
        row = f"  {name:28s}"
        for model_type in model_paths:
            mean, std = mean_and_std(result, model_type, metric)
            nonfinite = max(
                s["fraction_nonfinite"] for s in result["results"][model_type]
            )
            row += f" {model_type}: {mean:.3e} ± {std:.1e} (nonfinite {nonfinite:.1%})"
        row += f" Persistence: {mean_and_std(result, 'Persistence', metric)[0]:.3e}"
        print(row)

l2_rollout
  sine_box_submitted_grf       DPOT: 1.644e-01 ± 7.4e-03 (nonfinite 0.0%) HyperFluxFNOLocal: 4.134e-02 ± 1.2e-03 (nonfinite 0.0%) DISCO: 6.666e-02 ± 7.0e-03 (nonfinite 0.0%) Persistence: 1.153e-01
  sine_box_submitted_step      DPOT: 1.765e-01 ± 4.6e-03 (nonfinite 0.0%) HyperFluxFNOLocal: 6.068e-02 ± 2.0e-03 (nonfinite 0.0%) DISCO: 9.015e-02 ± 8.1e-03 (nonfinite 0.0%) Persistence: 1.397e-01
  sine_box_godunov_grf         DPOT: 1.644e-01 ± 7.4e-03 (nonfinite 0.0%) HyperFluxFNOLocal: 4.121e-02 ± 1.2e-03 (nonfinite 0.0%) DISCO: 6.660e-02 ± 7.0e-03 (nonfinite 0.0%) Persistence: 1.151e-01
  sine_box_godunov_step        DPOT: 1.765e-01 ± 4.6e-03 (nonfinite 0.0%) HyperFluxFNOLocal: 6.070e-02 ± 2.0e-03 (nonfinite 0.0%) DISCO: 9.016e-02 ± 8.1e-03 (nonfinite 0.0%) Persistence: 1.396e-01
  sine_rescaled_1-2_grf        DPOT: 1.630e-01 ± 5.6e-03 (nonfinite 0.0%) HyperFluxFNOLocal: 5.807e-02 ± 5.6e-04 (nonfinite 0.0%) DISCO: 8.113e-02 ± 1.9e-03 (nonfinite 0.0%) Persistence: 1.982e-01
  si

In [10]:
r = all_results["sine_rescaled_1-2_step"]
m = np.array(r["results"]["DPOT"][0]["l2_rollout"]["per_draw_window"])  # (draw, window)
print(np.argsort(np.nanmax(m, axis=1))[-5:], np.nanmedian(m), np.nanmax(m))

[12 84 87 44 18] 0.15096710622310638 995.8258056640625


## Fallback trigger

Fixed before the results existed (rebuttal plan, 2026-11-10): the variant with a' ~ U[-2, 2] is generated if the 20-step relative l2 error of HFluxNO exceeds half the persistence error on the bin |b| in (3, 4], for either initial-condition family.

In [ ]:
FALLBACK_RATIO = 0.5

for ic in ("grf", "step"):
    result = all_results[f"sine_rescaled_3-4_{ic}"]
    model_error, _ = mean_and_std(result, "HyperFluxFNOLocal", "l2_rollout")
    persistence_error, _ = mean_and_std(result, "Persistence", "l2_rollout")
    ratio = model_error / persistence_error
    print(
        f"{ic}: HFluxNO {model_error:.3e}, persistence {persistence_error:.3e}, "
        f"ratio {ratio:.2f} -> fallback {'triggered' if ratio > FALLBACK_RATIO else 'not triggered'}"
    )